In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

n = 10
modes = ["fast", "medium", "slow"]
sizes = ["top", "bottom"]
metric = "degree"

portfolios_dict = {}

for mode in modes:
    for size in sizes:
        portfolios_dict[f"{mode}_{size}_{n}_{metric}_portfolio"] = pd.read_parquet(f"../../data/06_portfolios/{mode}_{size}_{n}_{metric}_portfolio.parquet")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

df = pd.read_parquet("../../data/05_metrics/graph_metrics_df.parquet")

base_returns = "../../data/02_clean"
min_obs = 5

monthly_metrics = []

base_modes = "../../data/03_modes"

# Listar todos os arquivos returns_YYYY_MM.parquet
year_months = sorted([
    d for d in os.listdir(base_modes)
    if "_" in d and os.path.isdir(os.path.join(base_modes, d))
])

for ym in year_months:
    returns = pd.read_parquet(f"{base_returns}/returns_{ym}.parquet")

    # --- Monthly return (compound, preserving NaNs) ---
    monthly_return = (
        returns.apply(
            lambda x: (1 + x.dropna()).prod() - 1
            if x.notna().sum() >= min_obs else np.nan
        )
        .to_frame(name="monthly_return")
    )

    # --- Monthly volatility ---
    monthly_volatility = (
        returns.std()
        .to_frame(name="monthly_volatility")
    )

    # Merge
    df_month = (
        monthly_return
        .merge(monthly_volatility, left_index=True, right_index=True)
        .reset_index()
        .rename(columns={"index": "node"})
    )

    df_month["year_month"] = ym

    monthly_metrics.append(df_month)

# Concatenar tudo
monthly_metrics_df = pd.concat(monthly_metrics, ignore_index=True)

{'fast_top_10_degree_portfolio': Ticker          EDIT      PRPO      CODX      AEHL      PPBT      WORX  \
 Date                                                                     
 2019-02-01 -0.005982  0.000000 -0.044118 -0.088670 -0.070866  0.054545   
 2019-02-04  0.014352 -0.090426  0.015385 -0.016216 -0.008475 -0.167423   
 2019-02-05 -0.035144 -0.011696 -0.015152  0.005495  0.042735  0.956403   
 2019-02-06 -0.001892  0.017751 -0.023077 -0.016393 -0.008197 -0.025070   
 2019-02-07 -0.079147 -0.011628 -0.055118  0.072222  0.016529 -0.214286   
 ...              ...       ...       ...       ...       ...       ...   
 2024-12-23       NaN       NaN       NaN       NaN       NaN  0.012422   
 2024-12-24       NaN       NaN       NaN       NaN       NaN -0.030675   
 2024-12-26       NaN       NaN       NaN       NaN       NaN  0.088608   
 2024-12-27       NaN       NaN       NaN       NaN       NaN -0.023256   
 2024-12-30       NaN       NaN       NaN       NaN       NaN  0.077